# 04 · Baselines y criterios de aceptación (Fase 6)
Antes de modelar se define qué significa "funcionar".

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 30)
from src.config import *
df = pd.read_parquet(PROCESSED / "emovi_movilidad.parquet")
q1 = df[df.sigue_q1.notna()]
from src.models import train as T
tr, te = T.split(df)
tabla, modelos = T.compare(tr, te, T.NUM, T.CAT)
elegido, mejor_auc = T.choose(tabla)
modelo = modelos[elegido]

In [2]:
display(tabla[tabla.Modelo.str.startswith("Base")])

,Modelo,AUC CV (media),AUC CV (desv.),AUC prueba,Exactitud balanceada prueba,F1 prueba,Brier prueba
0,Base: siempre la clase mayoritaria,0.500,0.000,0.500,0.500,0.682,0.250
1,Base: sólo índice de origen,0.531,0.022,0.556,0.533,0.642,0.248


## Criterios de aceptación
| Métrica | Por qué | Umbral mínimo |
|---|---|---|
| **AUC** | Capacidad de ordenar riesgo; no depende del umbral | Superar a los dos baselines por ≥ 0.05 |
| Exactitud balanceada | Las clases están casi 50/50, pero se cuida igual | > 0.60 |
| Brier | Calibración de probabilidades (menor es mejor) | < 0.25 (baseline) |
| Desempeño por subgrupo | Que no falle en grupos vulnerables | AUC > 0.6 en grupos con n ≥ 100 |

**Lectura del baseline:** dentro del 20% más pobre, el índice de origen por sí solo casi no distingue quién sale (AUC ≈ 0.53–0.56). Las diferencias **dentro** del fondo de la distribución dependen de otras circunstancias.